In [ ]:
# Jupyter Notebook: Re-extract ibc_processed.csv and labels, then predict with KNN

# Cell 1: Import libraries
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from collections import Counter
import random  # for enrollment sampling

# Cell 2: Re-extract ibc_processed.csv from raw (with filtering and interpolation)
RAW_CSV = Path("data/raw/all_measurements.csv")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "ibc_processed.csv"

df_raw = pd.read_csv(RAW_CSV)
print(f"Raw shape: {df_raw.shape}")

# Drop NaNs
df = df_raw.dropna()
print(f"After dropna: {df.shape}")

# 3σ outlier removal
num = df.iloc[:, 1:]  # assume col 0 is subject_id
mask = (np.abs(num - num.mean()) <= 3 * num.std()).all(axis=1)
df = df[mask]
print(f"After 3σ filter: {df.shape}")

# Dynamic interpolation (raw has 71 spectral points)
n_raw = df.shape[1] - 1  # spectral columns
f_raw = np.linspace(50e3, 20e6, n_raw)
f_new = np.linspace(50e3, 20e6, 256)  # target 256 points

def interp_row(row):
    return interp1d(f_raw, row, kind="linear")(f_new)

spectra = np.vstack(df.iloc[:, 1:].apply(interp_row, axis=1))
spectra = (spectra - spectra.mean(1, keepdims=True)) / spectra.std(1, keepdims=True)  # z-score

pd.DataFrame(spectra, columns=[f"f_{i}" for i in range(256)]).to_csv(OUT_CSV, index=False)
print(f"Saved ibc_processed.csv → {OUT_CSV}")

# Cell 3: Re-extract filtered labels
LABEL_CSV = Path("data/labels_filtered.csv")

labels = df.iloc[:, 0]  # assume col 0 is subject_id
labels.name = "subject_id"
labels.to_csv(LABEL_CSV, index=False)
print(f"Saved labels_filtered.csv → {LABEL_CSV}")
print(f"Label distribution: {Counter(labels)}")

# Cell 4: Load data and run KNN with enrollment
X = pd.read_csv(OUT_CSV).values
y = pd.read_csv(LABEL_CSV)["subject_id"].values
print(f"X shape: {X.shape}, y shape: {y.shape}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
accs = []
for train_idx, test_idx in sgkf.split(X, y, groups=y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Enrollment: for each test subject, add 5 random samples to train
    test_subjects = np.unique(y_test)
    for sub in test_subjects:
        sub_idx = np.where(y_test == sub)[0]
        if len(sub_idx) < 6:  # need at least 5+1
            continue
        enroll_idx = random.sample(range(len(sub_idx)), 5)
        enroll_X = X_test[enroll_idx]
        enroll_y = y_test[enroll_idx]

        X_train = np.vstack((X_train, enroll_X))
        y_train = np.append(y_train, enroll_y)

        remain_idx = [i for i in range(len(sub_idx)) if i not in enroll_idx]
        X_test = X_test[remain_idx]
        y_test = y_test[remain_idx]

    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(X_train, y_train)
    acc = knn.score(X_test, y_test)
    accs.append(acc)
    print(f"Fold acc: {acc:.3f}, test classes: {len(set(y_test))}")

print(f"Average acc: {np.mean(accs):.3f}")


In [ ]:
# Jupyter Notebook: Re-extract ibc_processed.csv and labels, extract DWT features, then predict with KNN

# This notebook reimplements the preprocessing steps, generates filtered labels,
# extracts DWT features (Daubechies-4, level 2, stats per band), and runs 5-fold KNN evaluation.
# Copy-paste into Jupyter and run cell by cell.

# Cell 1: Import libraries
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from collections import Counter
import pywt  # for DWT

# Cell 2: Re-extract ibc_processed.csv from raw (with filtering and interpolation)
RAW_CSV = Path("data/raw/all_measurements.csv")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "ibc_processed.csv"

df_raw = pd.read_csv(RAW_CSV)
print(f"Raw shape: {df_raw.shape}")

# Drop NaNs
df = df_raw.dropna()
print(f"After dropna: {df.shape}")

# 3σ outlier removal
num = df.iloc[:, 1:]  # assume col 0 is subject_id
mask = (np.abs(num - num.mean()) <= 3 * num.std()).all(axis=1)
df = df[mask]
print(f"After 3σ filter: {df.shape}")

# Dynamic interpolation (raw has 71 spectral points)
n_raw = df.shape[1] - 1  # spectral columns
f_raw = np.linspace(50e3, 20e6, n_raw)
f_new = np.linspace(50e3, 20e6, 256)  # target 256 points

def interp_row(row):
    return interp1d(f_raw, row, kind="linear")(f_new)

spectra = np.vstack(df.iloc[:, 1:].apply(interp_row, axis=1))
spectra = (spectra - spectra.mean(1, keepdims=True)) / spectra.std(1, keepdims=True)  # z-score

pd.DataFrame(spectra, columns=[f"f_{i}" for i in range(256)]).to_csv(OUT_CSV, index=False)
print(f"Saved ibc_processed.csv → {OUT_CSV}")

# Cell 3: Re-extract filtered labels
LABEL_CSV = Path("data/labels_filtered.csv")

labels = df.iloc[:, 0]  # assume col 0 is subject_id
labels.name = "subject_id"
labels.to_csv(LABEL_CSV, index=False)
print(f"Saved labels_filtered.csv → {LABEL_CSV}")
print(f"Label distribution: {Counter(labels)}")

# Cell 4: Extract DWT features (Daubechies-4, level 2, stats per band)
DWT_CSV = Path("features/dwt_features.csv")
DWT_CSV.parent.mkdir(parents=True, exist_ok=True)

LEVEL = 2
WAVELET = "db4"

def dwt_stats(coeff):
    energy = np.sum(coeff**2)
    p = coeff**2 / energy if energy > 0 else np.zeros_like(coeff)
    entropy = -np.sum(p * np.log2(p + 1e-12))
    mean = np.mean(coeff)
    std = np.std(coeff)
    return [energy, entropy, mean, std]

dwt_feats = []
for row in spectra:
    coeffs = pywt.wavedec(row, WAVELET, level=LEVEL, mode="periodization")
    feats = [s for c in coeffs for s in dwt_stats(c)]
    dwt_feats.append(feats)

dwt_df = pd.DataFrame(dwt_feats, columns=[f"b{b}_{s}" for b in range(LEVEL+1) for s in ("energy", "entropy", "mean", "std")])
dwt_df.to_csv(DWT_CSV, index=False)
print(f"Saved DWT features → {DWT_CSV}")

# Cell 5: Load DWT features and run KNN with enrollment
X = pd.read_csv(DWT_CSV).values
y = pd.read_csv(LABEL_CSV)["subject_id"].values
print(f"X shape: {X.shape}, y shape: {y.shape}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
accs = []
for train_idx, test_idx in sgkf.split(X, y, groups=y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Enrollment: for each test subject, add 5 random samples to train
    test_subjects = np.unique(y_test)
    for sub in test_subjects:
        sub_idx = np.where(y_test == sub)[0]
        if len(sub_idx) < 6:  # need at least 5+1
            continue
        enroll_idx = np.random.choice(sub_idx, 5, replace=False)
        enroll_X = X_test[enroll_idx]
        enroll_y = y_test[enroll_idx]

        X_train = np.vstack((X_train, enroll_X))
        y_train = np.append(y_train, enroll_y)

        remain_idx = np.setdiff1d(sub_idx, enroll_idx)
        X_test = X_test[remain_idx]
        y_test = y_test[remain_idx]

    knn = KNeighborsClassifier(n_neighbors=1)
    knn.fit(X_train, y_train)
    acc = knn.score(X_test, y_test)
    accs.append(acc)
    print(f"Fold acc: {acc:.3f}, test classes: {len(set(y_test))}")

print(f"Average acc: {np.mean(accs):.3f}")


In [ ]:
# Jupyter Notebook: Re-extract ibc_processed.csv and labels, extract DWT features, then predict with Random Forest

# Cell 1: Import libraries
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from collections import Counter
import pywt  # for DWT
import random  # for enrollment sampling

# Cell 2: Re-extract ibc_processed.csv from raw (with filtering and interpolation)
RAW_CSV = Path("data/raw/all_measurements.csv")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV = OUT_DIR / "ibc_processed.csv"

df_raw = pd.read_csv(RAW_CSV)
print(f"Raw shape: {df_raw.shape}")

# Drop NaNs
df = df_raw.dropna()
print(f"After dropna: {df.shape}")

# 3σ outlier removal
num = df.iloc[:, 1:]  # assume col 0 is subject_id
mask = (np.abs(num - num.mean()) <= 3 * num.std()).all(axis=1)
df = df[mask]
print(f"After 3σ filter: {df.shape}")

# Dynamic interpolation (raw has 71 spectral points)
n_raw = df.shape[1] - 1  # spectral columns
f_raw = np.linspace(50e3, 20e6, n_raw)
f_new = np.linspace(50e3, 20e6, 256)  # target 256 points

def interp_row(row):
    return interp1d(f_raw, row, kind="linear")(f_new)

spectra = np.vstack(df.iloc[:, 1:].apply(interp_row, axis=1))
spectra = (spectra - spectra.mean(1, keepdims=True)) / spectra.std(1, keepdims=True)  # z-score

pd.DataFrame(spectra, columns=[f"f_{i}" for i in range(256)]).to_csv(OUT_CSV, index=False)
print(f"Saved ibc_processed.csv → {OUT_CSV}")

# Cell 3: Re-extract filtered labels
LABEL_CSV = Path("data/labels_filtered.csv")

labels = df.iloc[:, 0]  # assume col 0 is subject_id
labels.name = "subject_id"
labels.to_csv(LABEL_CSV, index=False)
print(f"Saved labels_filtered.csv → {LABEL_CSV}")
print(f"Label distribution: {Counter(labels)}")

# Cell 4: Extract DWT features (Daubechies-4, level 2, stats per band)
DWT_CSV = Path("features/dwt_features.csv")
DWT_CSV.parent.mkdir(parents=True, exist_ok=True)

LEVEL = 2
WAVELET = "db4"

def dwt_stats(coeff):
    energy = np.sum(coeff**2)
    p = coeff**2 / energy if energy > 0 else np.zeros_like(coeff)
    entropy = -np.sum(p * np.log2(p + 1e-12))
    mean = np.mean(coeff)
    std = np.std(coeff)
    return [energy, entropy, mean, std]

dwt_feats = []
for row in spectra:
    coeffs = pywt.wavedec(row, WAVELET, level=LEVEL, mode="periodization")
    feats = [s for c in coeffs for s in dwt_stats(c)]
    dwt_feats.append(feats)

dwt_df = pd.DataFrame(dwt_feats, columns=[f"b{b}_{s}" for b in range(LEVEL+1) for s in ("energy", "entropy", "mean", "std")])
dwt_df.to_csv(DWT_CSV, index=False)
print(f"Saved DWT features → {DWT_CSV}")

# Cell 5: Load DWT features and run RF with enrollment
X = pd.read_csv(DWT_CSV).values
y = pd.read_csv(LABEL_CSV)["subject_id"].values
print(f"X shape: {X.shape}, y shape: {y.shape}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
accs = []
for train_idx, test_idx in sgkf.split(X, y, groups=y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Enrollment: for each test subject, add 5 random samples to train
    test_subjects = np.unique(y_test)
    for sub in test_subjects:
        sub_idx = np.where(y_test == sub)[0]
        if len(sub_idx) < 6:  # need at least 5+1
            continue
        enroll_idx = np.random.choice(sub_idx, 5, replace=False)
        enroll_X = X_test[enroll_idx]
        enroll_y = y_test[enroll_idx]

        X_train = np.vstack((X_train, enroll_X))
        y_train = np.append(y_train, enroll_y)

        remain_idx = np.setdiff1d(sub_idx, enroll_idx)
        X_test = X_test[remain_idx]
        y_test = y_test[remain_idx]

    rf = RandomForestClassifier(n_estimators=500, random_state=42)
    rf.fit(X_train, y_train)
    acc = rf.score(X_test, y_test)
    accs.append(acc)
    print(f"Fold acc: {acc:.3f}, test classes: {len(set(y_test))}")

print(f"Average acc: {np.mean(accs):.3f}")


In [ ]:
import numpy as np
import pandas as pd
import pywt
from scipy.interpolate import interp1d
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
import random

RAW_CSV = Path('data/raw/all_measurements.csv')
PROCESSED_CSV = Path('data/processed/ibc_processed.csv')
LABELS_CSV = Path('data/labels_filtered.csv')
DWT_FEAT_CSV = Path('data/dwt_features.csv')

# Load raw data
raw = pd.read_csv(RAW_CSV)

raw = raw.dropna()

# 3 sigma filter
cols = raw.columns[1:]
mask = (np.abs(raw[cols] - raw[cols].mean()) <= 3 * raw[cols].std()).all(axis=1)
raw = raw.loc[mask]

# Save labels
labels = raw.iloc[:, [0]]
labels.columns = ['subject_id']
labels.to_csv(LABELS_CSV, index=False)

# Interpolate spectra
n_raw = len(cols)
freq_orig = np.linspace(50e3, 20e6, n_raw)
freq_new = np.linspace(50e3, 20e6, 256)

def interp_spectrum(row):
    return interp1d(freq_orig, row, kind='linear')(freq_new)

spectra = np.vstack(raw.iloc[:, 1:].apply(interp_spectrum, axis=1))
# z-score
spectra = (spectra - np.mean(spectra, axis=1, keepdims=True)) / np.std(spectra, axis=1, keepdims=True)
pd.DataFrame(spectra).to_csv(PROCESSED_CSV, index=False)

# Extract DWT features
level = 2
wavelet = 'db4'


def compute_stats(coeff):
    energy = np.sum(coeff ** 2)
    if energy > 0:
        p = coeff ** 2 / energy
    else:
        p = np.zeros_like(coeff)
    entropy = -np.sum(p * np.log2(p + 1e-12))
    return [energy, entropy, np.mean(coeff), np.std(coeff)]

feat_list = []
for row in spectra:
    coeffs = pywt.wavedec(row, wavelet, level=level, mode='periodization')
    feat_list.append([stat for coeff in coeffs for stat in compute_stats(coeff)])

columns = [f'b{b}_{stat}' for b in range(level + 1) for stat in ['energy', 'entropy', 'mean', 'std']]
feat_df = pd.DataFrame(feat_list, columns=columns)
feat_df.to_csv(DWT_FEAT_CSV, index=False)

# Training with LightGBM and enrollment
X = feat_df.values
y = labels['subject_id'].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
accs = []

for train_idx, test_idx in sgkf.split(X, y, groups=y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Enrollment - add 5 samples from each test subject to training
    for subject in np.unique(y_test):
        indices = np.where(y_test == subject)[0]
        if len(indices) < 6:
            continue
        enroll_indices = np.random.choice(indices, 5, replace=False)
        X_train = np.vstack([X_train, X_test[enroll_indices]])
        y_train = np.concatenate([y_train, y_test[enroll_indices]])
        keep_indices = np.array([i for i in range(len(y_test)) if i not in enroll_indices])
        X_test = X_test[keep_indices]
        y_test = y_test[keep_indices]

    clf = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42)
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    accs.append(acc)
    print(f'Fold accuracy: {acc:.3f}, test subjects: {len(np.unique(y_test))}')

print(f'Average accuracy: {np.mean(accs):.3f}')


In [ ]:
import numpy as np
import pandas as pd
import pywt
from scipy.interpolate import interp1d
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
import random

RAW_CSV = Path('data/raw/all_measurements.csv')
PROCESSED_CSV = Path('data/processed/ibc_processed.csv')
LABELS_CSV = Path('data/labels_filtered.csv')
DWT_FEAT_CSV = Path('data/dwt_features.csv')

# Load raw data
raw = pd.read_csv(RAW_CSV)

raw = raw.dropna()

# 3 sigma filter
cols = raw.columns[1:]
mask = (np.abs(raw[cols] - raw[cols].mean()) <= 3 * raw[cols].std()).all(axis=1)
raw = raw.loc[mask]

# Save labels
labels = raw.iloc[:, [0]]
labels.columns = ['subject_id']
labels.to_csv(LABELS_CSV, index=False)

# Interpolate spectra
n_raw = len(cols)
freq_orig = np.linspace(50e3, 20e6, n_raw)
freq_new = np.linspace(50e3, 20e6, 256)

def interp_spectrum(row):
    return interp1d(freq_orig, row, kind='linear')(freq_new)

spectra = np.vstack(raw.iloc[:, 1:].apply(interp_spectrum, axis=1))
# z-score
spectra = (spectra - np.mean(spectra, axis=1, keepdims=True)) / np.std(spectra, axis=1, keepdims=True)
pd.DataFrame(spectra).to_csv(PROCESSED_CSV, index=False)

# Extract DWT features
level = 2
wavelet = 'db4'


def compute_stats(coeff):
    energy = np.sum(coeff ** 2)
    if energy > 0:
        p = coeff ** 2 / energy
    else:
        p = np.zeros_like(coeff)
    entropy = -np.sum(p * np.log2(p + 1e-12))
    return [energy, entropy, np.mean(coeff), np.std(coeff)]

feat_list = []
for row in spectra:
    coeffs = pywt.wavedec(row, wavelet, level=level, mode='periodization')
    feat_list.append([stat for coeff in coeffs for stat in compute_stats(coeff)])

columns = [f'b{b}_{stat}' for b in range(level + 1) for stat in ['energy', 'entropy', 'mean', 'std']]
feat_df = pd.DataFrame(feat_list, columns=columns)
feat_df.to_csv(DWT_FEAT_CSV, index=False)

# Training with LightGBM and enrollment
X = feat_df.values
y = labels['subject_id'].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
accs = []

for train_idx, test_idx in sgkf.split(X, y, groups=y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Enrollment - add 5 samples from each test subject to training
    for subject in np.unique(y_test):
        indices = np.where(y_test == subject)[0]
        if len(indices) < 6:
            continue
        enroll_indices = np.random.choice(indices, 5, replace=False)
        X_train = np.vstack([X_train, X_test[enroll_indices]])
        y_train = np.concatenate([y_train, y_test[enroll_indices]])
        keep_indices = np.array([i for i in range(len(y_test)) if i not in enroll_indices])
        X_test = X_test[keep_indices]
        y_test = y_test[keep_indices]

    clf = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42)
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    accs.append(acc)
    print(f'Fold accuracy: {acc:.3f}, test subjects: {len(np.unique(y_test))}')

print(f'Average accuracy: {np.mean(accs):.3f}')


In [ ]:
import numpy as np
import pandas as pd
import pywt
from scipy.interpolate import interp1d
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier
import random

RAW_CSV = Path('data/raw/all_measurements.csv')
PROCESSED_CSV = Path('data/processed/ibc_processed.csv')
LABELS_CSV = Path('data/labels_filtered.csv')
DWT_FEAT_CSV = Path('data/dwt_features.csv')

# Load raw data
raw = pd.read_csv(RAW_CSV)

raw = raw.dropna()

# 3 sigma filter
cols = raw.columns[1:]
mask = (np.abs(raw[cols] - raw[cols].mean()) <= 3 * raw[cols].std()).all(axis=1)
raw = raw.loc[mask]

# Save labels
labels = raw.iloc[:, [0]]
labels.columns = ['subject_id']
labels.to_csv(LABELS_CSV, index=False)

# Interpolate spectra
n_raw = len(cols)
freq_orig = np.linspace(50e3, 20e6, n_raw)
freq_new = np.linspace(50e3, 20e6, 256)

def interp_spectrum(row):
    return interp1d(freq_orig, row, kind='linear')(freq_new)

spectra = np.vstack(raw.iloc[:, 1:].apply(interp_spectrum, axis=1))
# z-score
spectra = (spectra - np.mean(spectra, axis=1, keepdims=True)) / np.std(spectra, axis=1, keepdims=True)
pd.DataFrame(spectra).to_csv(PROCESSED_CSV, index=False)

# Extract DWT features
level = 2
wavelet = 'db4'


def compute_stats(coeff):
    energy = np.sum(coeff ** 2)
    if energy > 0:
        p = coeff ** 2 / energy
    else:
        p = np.zeros_like(coeff)
    entropy = -np.sum(p * np.log2(p + 1e-12))
    return [energy, entropy, np.mean(coeff), np.std(coeff)]

feat_list = []
for row in spectra:
    coeffs = pywt.wavedec(row, wavelet, level=level, mode='periodization')
    feat_list.append([stat for coeff in coeffs for stat in compute_stats(coeff)])

columns = [f'b{b}_{stat}' for b in range(level + 1) for stat in ['energy', 'entropy', 'mean', 'std']]
feat_df = pd.DataFrame(feat_list, columns=columns)
feat_df.to_csv(DWT_FEAT_CSV, index=False)

# Training with LightGBM and enrollment
X = feat_df.values
y = labels['subject_id'].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
accs = []

for train_idx, test_idx in sgkf.split(X, y, groups=y):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Enrollment - add 5 samples from each test subject to training
    for subject in np.unique(y_test):
        indices = np.where(y_test == subject)[0]
        if len(indices) < 6:
            continue
        enroll_indices = np.random.choice(indices, 5, replace=False)
        X_train = np.vstack([X_train, X_test[enroll_indices]])
        y_train = np.concatenate([y_train, y_test[enroll_indices]])
        keep_indices = np.array([i for i in range(len(y_test)) if i not in enroll_indices])
        X_test = X_test[keep_indices]
        y_test = y_test[keep_indices]

    clf = LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=7, random_state=42)
    clf.fit(X_train, y_train)
    acc = clf.score(X_test, y_test)
    accs.append(acc)
    print(f'Fold accuracy: {acc:.3f}, test subjects: {len(np.unique(y_test))}')

print(f'Average accuracy: {np.mean(accs):.3f}')
